# 01 — Annotation

Generate a standardized transcript annotation table from the reference annotation for downstream Ribo-seq analyses.

## Purpose

This notebook extracts transcript-level annotations from the reference genome annotation.

The resulting table serves as the backbone for all downstream analyses, including RNA quantification, Ribo-seq quantification, translation efficiency calculation, alignment summaries, and footprint analyses.

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import gzip
from collections import Counter

In [5]:
# ============================================
# Configuration
# ============================================

# Workflow repository
REPO = Path.home() / "Code" / "processing" / "riboseq"

# Reference resources
REFERENCE = REPO / "refs" / "built" / "arabidopsis_tair10_rsemclean"

# Project data
PROJECT = Path("/mnt/d/Ibnu/riboseq/AT")

# nf-core results
INPUT = PROJECT / "nfcore" / "10pct"

# Generated tables
TABLES = PROJECT / "TABLES"

# Optional figures
FIGURES = PROJECT / "FIGURES"

TABLES.mkdir(exist_ok=True)
FIGURES.mkdir(exist_ok=True)

In [9]:
# ============================================
# Input inspection
# ============================================

print(f"Repository       : {REPO}")
print(f"Reference        : {REFERENCE}")
print(f"Project directory: {PROJECT}")
print(f"Input directory  : {INPUT}")
print(f"Tables directory : {TABLES}")
print(f"Figures directory: {FIGURES}")

print()
print("Reference exists:", REFERENCE.exists())
print("Input exists    :", INPUT.exists())
print("Tables exists   :", TABLES.exists())
print("Figures exists  :", FIGURES.exists())

Repository       : /home/ha-ibnu/Code/processing/riboseq
Reference        : /home/ha-ibnu/Code/processing/riboseq/refs/built/arabidopsis_tair10_rsemclean
Project directory: /mnt/d/Ibnu/riboseq/AT
Input directory  : /mnt/d/Ibnu/riboseq/AT/nfcore/10pct
Tables directory : /mnt/d/Ibnu/riboseq/AT/TABLES
Figures directory: /mnt/d/Ibnu/riboseq/AT/FIGURES

Reference exists: True
Input exists    : True
Tables exists   : True
Figures exists  : True


## Annotation table design

The target output of this notebook is:

| Column | Description |
|--------|-------------|
| `Transcript_ID` | Transcript identifier |
| `Gene_ID` | Gene identifier |
| `Chr` | Chromosome or contig |
| `Strand` | Genomic strand |
| `Transcript_start` | Transcript start coordinate |
| `Transcript_end` | Transcript end coordinate |
| `Transcript_length` | Transcript length |
| `CDS_length` | Total CDS length |
| `UTR5_length` | Total 5′UTR length |
| `UTR3_length` | Total 3′UTR length |

In [11]:
# ============================================
# Load reference annotation
# ============================================
GTF = REFERENCE / "at.rsem.gtf"

print("Reference annotation")
print("--------------------")
print(GTF)
print("Exists:", GTF.exists())

Reference annotation
--------------------
/home/ha-ibnu/Code/processing/riboseq/refs/built/arabidopsis_tair10_rsemclean/at.rsem.gtf
Exists: True


In [12]:
with open(GTF) as f:
    for _ in range(20):
        print(f.readline().rstrip())

Chr1	Araport11	transcript	3631	5899	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010"
Chr1	Araport11	exon	3631	3913	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	3996	4276	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	4486	4605	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	4706	5095	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	5174	5326	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	exon	5439	5899	.	+	.	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	3760	3913	.	+	0	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	3996	4276	.	+	2	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	4486	4605	.	+	0	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	4706	5095	.	+	0	transcript_id "AT1G01010.1"; gene_id "AT1G01010";
Chr1	Araport11	CDS	5174	5326	.	+	0	transcript_id "AT1

In [13]:
# ============================================
# Inspect feature types
# ============================================

feature_counts = {}

with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue
        fields = line.rstrip("\n").split("\t")
        if len(fields) < 9:
            continue
        feature = fields[2]
        feature_counts[feature] = feature_counts.get(feature, 0) + 1

pd.Series(feature_counts).sort_values(ascending=False)

exon          322737
CDS           286355
transcript     59478
dtype: int64

In [15]:

feature_counts = Counter()

with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip().split("\t")

        if len(fields) != 9:
            continue

        feature_counts[fields[2]] += 1

pd.DataFrame(
    feature_counts.items(),
    columns=["Feature", "Count"]
).sort_values("Count", ascending=False).reset_index(drop=True)

,Feature,Count
0,exon,322737
1,CDS,286355
2,transcript,59478


In [16]:
# ============================================
# Inspect annotation attributes
# ============================================

with open(GTF) as f:
    for line in f:
        if line.startswith("#"):
            continue

        fields = line.rstrip().split("\t")

        print(fields[8])

        break

transcript_id "AT1G01010.1"; gene_id "AT1G01010"


In [17]:
# ============================================
# Attribute parser
# ============================================

def parse_gtf_attributes(attr: str) -> dict:
    """
    Parse a GTF attribute string.

    Example:
    transcript_id "AT1G01010.1"; gene_id "AT1G01010";
    """
    result = {}

    for item in attr.strip().split(";"):
        item = item.strip()

        if not item:
            continue

        key, value = item.split(" ", 1)
        result[key] = value.strip().strip('"')

    return result


# Test parser
example_attr = 'transcript_id "AT1G01010.1"; gene_id "AT1G01010";'
parse_gtf_attributes(example_attr)

{'transcript_id': 'AT1G01010.1', 'gene_id': 'AT1G01010'}